# Confirmatory Run: Pair Jitter+Grayscale (800 epochs, 3 seeds)
Validates the screening finding that color-only augmentations (no spatial structure) collapse to 26.45%.

| Setting | Value |
|---|---|
| Epochs | 800 |
| Seeds | 42, 43, 44 |
| Batch size | 512 |
| Augmentations | ColorJitter + RandomGrayscale only |

**Expected runtime:** ~4h × 3 seeds = ~12h on A100 (PSC).

In [ ]:
import torch
print(f'PyTorch : {torch.__version__}')
print(f'GPU     : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE"}')
assert torch.cuda.is_available(), 'No GPU detected.'

In [ ]:
import os, subprocess

REPO_URL   = 'https://github.com/SAlaMusa/IDL.git'
REPO_DIR   = '/kaggle/working/IDL'
OUTPUT_DIR = '/kaggle/working'

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)

os.chdir(REPO_DIR)
print('Working dir:', os.getcwd())

In [ ]:
import glob, shutil, csv, datetime, statistics

CONFIG    = 'configs/pair_jitter_grayscale.yaml'
EPOCHS    = 800
BATCH     = 512
SEEDS     = [42, 43, 44]
RESULTS   = os.path.join(OUTPUT_DIR, 'confirmatory_pair_jitter_grayscale_results.csv')

if not os.path.exists(RESULTS):
    with open(RESULTS, 'w', newline='') as f:
        csv.writer(f).writerow(['exp_name', 'seed', 'best_top1', 'checkpoint', 'timestamp'])

top1_scores = []

for seed in SEEDS:
    exp_name  = f'confirmatory_pair_jitter_grayscale_seed{seed}'
    ckpt_path = os.path.join(OUTPUT_DIR, f'{exp_name}_ep{EPOCHS}.pth.tar')

    if os.path.exists(ckpt_path):
        print(f'[seed {seed}] Checkpoint exists, skipping pretraining.')
    else:
        print(f"\n{'='*60}\n[seed {seed}] PRETRAINING\n{'='*60}")
        subprocess.run([
            'python', 'run.py', '--config', CONFIG,
            '--epochs', str(EPOCHS), '--batch-size', str(BATCH),
            '-j', '2', '--seed', str(seed), '--fp16-precision',
        ], check=True)
        latest = max(glob.glob('runs/**/*.pth.tar', recursive=True), key=os.path.getmtime)
        shutil.copy2(latest, ckpt_path)
        print(f'[seed {seed}] Checkpoint saved: {ckpt_path}')

    print(f'\n[seed {seed}] LINEAR EVALUATION')
    subprocess.run([
        'python', 'linear_eval.py',
        '--checkpoint', ckpt_path, '--dataset', 'cifar10',
        '--arch', 'resnet18', '--epochs', '100',
        '-b', '256', '-j', '2', '--seed', str(seed),
    ], check=True)

    with open('linear_eval_results.csv') as f:
        best_top1 = float(list(csv.DictReader(f))[-1]['best_top1'])

    top1_scores.append(best_top1)
    with open(RESULTS, 'a', newline='') as f:
        csv.writer(f).writerow([exp_name, seed, f'{best_top1:.2f}', ckpt_path,
                                datetime.datetime.now().strftime('%Y-%m-%d %H:%M')])
    print(f'[seed {seed}] Best Top-1: {best_top1:.2f}%')

print(f'\n=== Confirmatory Pair Jitter+Grayscale Complete ===')
for seed, acc in zip(SEEDS, top1_scores):
    print(f'  seed {seed}: {acc:.2f}%')
print(f'  Mean: {statistics.mean(top1_scores):.2f}%  Std: {statistics.stdev(top1_scores):.2f}%')
print(f'\nResults saved to: {RESULTS}')